# Feature Distillation — なぜ中間特徴量は最終出力より有効か

ここでは以下の問いに答える：

> **なぜ中間特徴量（feature map）を使った知識蒸留は、最終出力（logit）だけを使うより性能が高いのか？**

## 実験設定

| 項目 | 内容 |
|---|---|
| データ | `sklearn digits` (8×8 グレー画像、10クラス、1,797枚) |
| 教師 CNN | ch1=32, ch2=64 — 特徴マップ 64ch×8×8 = **4,096次元** |
| 生徒 CNN | ch1=8,  ch2=16 — 特徴マップ 16ch×8×8 = **1,024次元** |
| 生徒ラベル数 | **150枚** のみ（CE 損失に使用）。残り 1,197枚 は KD/特徴蒸留に使用 |

## 比較する5手法

| 手法 | ロジット蒸留 | 特徴量蒸留 |
|---|---|---|
| Scratch | — | — |
| Logit KD (Hinton) | KL(soft q\|\|soft p), T=4 | — |
| DKD (Decoupled KD) | TCKD + 2×NCKD | — |
| FitNets + KD | KD | MSE(Adapter(F_s), F_t) |
| AT + KD | KD | MSE(Attn(F_s), Attn(F_t)) |


In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
np.random.seed(42)
print("PyTorch", torch.__version__)


PyTorch 2.12.0+cu130


## 1. データセット

学習プール全体（~1,347枚）は KD・特徴蒸留のターゲット生成に使う。
生徒の CE 損失は 150枚 のラベル付きサンプルにのみ適用する。
これが蒸留が"情報を追加"するレジームになる。


In [2]:
digits = load_digits()
X = digits.images.astype('float32') / 16.0   # [1797, 8, 8], 正規化
X = X[:, None]                                 # [1797, 1, 8, 8]
y = digits.target.astype('int64')

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
X_lab, _, y_lab, _ = train_test_split(
    X_tr, y_tr, train_size=150, random_state=42, stratify=y_tr
)

X_pool_t = torch.tensor(X_tr);  y_pool_t = torch.tensor(y_tr)
X_te_t   = torch.tensor(X_te);  y_te_t   = torch.tensor(y_te)
X_lab_t  = torch.tensor(X_lab); y_lab_t  = torch.tensor(y_lab)

pool_loader = DataLoader(TensorDataset(X_pool_t, y_pool_t), batch_size=64, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_te_t,  y_te_t),  batch_size=256)
lab_loader   = DataLoader(TensorDataset(X_lab_t, y_lab_t), batch_size=32, shuffle=True)

print(f"学習プール : {len(X_tr):4d} 枚  (KD/特徴蒸留に全数使用)")
print(f"生徒ラベル : {len(X_lab):4d} 枚  (CE 損失のみ)")
print(f"テスト     : {len(X_te):4d} 枚")


学習プール : 1347 枚  (KD/特徴蒸留に全数使用)
生徒ラベル :  150 枚  (CE 損失のみ)
テスト     :  450 枚


## 2. モデル定義

```
教師: Conv(1→32) → Conv(32→64) → Pool → Linear → 10
生徒: Conv(1→8)  → Conv(8→16)  → Pool → Linear → 10
```

教師の特徴マップ = 64ch × 8 × 8 = **4,096次元**
生徒の特徴マップ = 16ch × 8 × 8 = **1,024次元**
最終ロジット = わずか **10次元**


In [3]:
class CNN(nn.Module):
    def __init__(self, ch1, ch2, num_classes=10):
        super().__init__()
        self.feat_ch = ch2
        self.b1 = nn.Sequential(
            nn.Conv2d(1, ch1, 3, padding=1), nn.BatchNorm2d(ch1), nn.ReLU()
        )
        self.b2 = nn.Sequential(
            nn.Conv2d(ch1, ch2, 3, padding=1), nn.BatchNorm2d(ch2), nn.ReLU()
        )
        self.pool = nn.MaxPool2d(2)          # 8×8 → 4×4
        self.head = nn.Linear(ch2 * 4 * 4, num_classes)

    def forward(self, x):
        x    = self.b1(x)
        feat = self.b2(x)                    # [B, ch2, 8, 8]
        logits = self.head(self.pool(feat).flatten(1))
        return logits, feat

def make_teacher(): return CNN(32, 64)
def make_student(): return CNN( 8, 16)

t_params = sum(p.numel() for p in make_teacher().parameters())
s_params = sum(p.numel() for p in make_student().parameters())
print(f"教師パラメータ数 : {t_params:,}")
print(f"生徒パラメータ数 : {s_params:,}  (教師の {t_params/s_params:.1f}× 少ない)")


教師パラメータ数 : 29,258
生徒パラメータ数 : 3,866  (教師の 7.6× 少ない)


## 3. なぜ中間特徴量がロジットより有効か — 直観

### 情報次元数の比較

| 信号 | 次元数 | 含む情報 |
|---|---|---|
| 最終ロジット | **10** | クラス間の相対確率（"4らしさ" vs "9らしさ"） |
| 教師の特徴マップ | **4,096** | どこを見ているか（空間）＋何が重要か（チャネル）＋強度 |

### 勾配パスの違い

```
特徴蒸留:   loss_feat → Conv最終層 → Conv中間層  ← 全層に直接シグナル
ロジット蒸留: loss_KD → Linear(head) → ボトルネック(10値) → 上流層（薄いシグナル）
```

以下で実際に 教師の特徴マップの "情報量" を確認する。


In [4]:
# 教師を仮に1エポック学習して特徴マップを可視化
_t_tmp = make_teacher()
_t_tmp.eval()

xb_vis, yb_vis = next(iter(test_loader))
with torch.no_grad():
    t_logits_vis, t_feat_vis = _t_tmp(xb_vis[:1])

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

# (A) ロジット: 10次元
axes[0].bar(range(10), torch.softmax(t_logits_vis[0], 0).numpy(),
            color='steelblue', alpha=0.8)
axes[0].set_xlabel('クラス'); axes[0].set_ylabel('Softmax 確率')
axes[0].set_title(f'(A) ロジット — {t_logits_vis.shape[1]} 次元\n最終出力（ランダム初期化時）')
axes[0].set_xticks(range(10))

# (B) 特徴マップの分布: 4096次元
feat_flat = t_feat_vis[0].flatten().numpy()
axes[1].hist(feat_flat, bins=40, color='tomato', alpha=0.8)
axes[1].set_xlabel('活性値'); axes[1].set_ylabel('頻度')
axes[1].set_title(f'(B) 特徴マップ値の分布\n{t_feat_vis[0].numel()} 次元（{t_feat_vis.shape[1]}ch × 8 × 8）')

# (C) 特徴マップの空間構造（最初のチャネル 4 枚）
grid = t_feat_vis[0, :4].numpy()
im = axes[2].imshow(
    np.concatenate([grid[i] for i in range(4)], axis=1),
    cmap='RdBu', aspect='auto'
)
axes[2].set_title('(C) 特徴マップ（最初の4チャネル）\n空間的な"注目場所"が見える')
axes[2].axis('off')
plt.colorbar(im, ax=axes[2], fraction=0.046)

plt.suptitle('ロジット（10次元）vs 特徴マップ（4,096次元）', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('/tmp/info_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"\nロジット次元数: {t_logits_vis.shape[1]}")
print(f"特徴マップ次元数: {t_feat_vis[0].numel()}  ({t_feat_vis.shape[1]}ch × {t_feat_vis.shape[2]} × {t_feat_vis.shape[3]})")
print(f"情報量の比: {t_feat_vis[0].numel() / t_logits_vis.shape[1]:.0f}×")
del _t_tmp



ロジット次元数: 10
特徴マップ次元数: 4096  (64ch × 8 × 8)
情報量の比: 410×


/tmp/ipykernel_44998/1783897201.py:35: UserWarning: Glyph 12463 (\N{KATAKANA LETTER KU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/1783897201.py:35: UserWarning: Glyph 12521 (\N{KATAKANA LETTER RA}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/1783897201.py:35: UserWarning: Glyph 12473 (\N{KATAKANA LETTER SU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/1783897201.py:35: UserWarning: Glyph 30906 (\N{CJK UNIFIED IDEOGRAPH-78BA}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/1783897201.py:35: UserWarning: Glyph 29575 (\N{CJK UNIFIED IDEOGRAPH-7387}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/1783897201.py:35: UserWarning: Glyph 12525 (\N{KATAKANA LETTER RO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/1783897201.py:35: UserWarning: Glyph 12472 (\N{KATAKANA LETTER ZI}) missing from font(s) DejaVu San

## 4. 教師モデルの学習


In [5]:
@torch.no_grad()
def evaluate(model):
    model.eval()
    correct = total = 0
    for xb, yb in test_loader:
        logits, _ = model(xb)
        correct += (logits.argmax(1) == yb).sum().item()
        total += len(yb)
    return correct / total

def train_teacher(epochs=80):
    torch.manual_seed(1)
    t = make_teacher()
    opt = torch.optim.AdamW(t.parameters(), lr=2e-3, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    for _ in range(epochs):
        t.train()
        for xb, yb in pool_loader:
            opt.zero_grad()
            F.cross_entropy(t(xb)[0], yb).backward()
            opt.step()
        sch.step()
    return t

teacher = train_teacher()
teacher.eval()
for p in teacher.parameters():
    p.requires_grad_(False)
acc_teacher = evaluate(teacher)
print(f"教師 テスト精度: {acc_teacher:.4f}")


教師 テスト精度: 0.9911


## 5. 損失関数の定義

### Hinton KD（温度付き KL ダイバージェンス）

$$\mathcal{L}_{\text{KD}} = T^2 \cdot \text{KL}\!\left(\sigma(z_s/T)\,\Vert\,\sigma(z_t/T)\right)$$

$T^2$ 係数は、温度で softmax が縮小する勾配スケールを補正するために必要。

### DKD（Decoupled KD, CVPR 2022）

KD 損失を **TCKD**（ターゲットクラス vs 全残余）と **NCKD**（非ターゲット間の分布）に分離：

$$\mathcal{L}_{\text{DKD}} = \alpha \cdot \text{TCKD} + \beta \cdot \text{NCKD}$$

標準 KD では $\beta = 1 - p_t^T$ となり、教師が確信しているサンプルほど NCKD が抑制される。
DKD は $\beta > 1$ にすることで dark knowledge（"非ターゲット間の類似性"）を強化する。

### FitNets（特徴 MSE）

$$\mathcal{L}_{\text{FitNets}} = \frac{1}{2}\|\,r(F_s) - F_t\,\|_2^2$$

$r$: 1×1 conv アダプター（生徒の ch 16 → 教師の ch 64）。**推論時は捨てる**。

### AT（Attention Transfer）

$$A(F) = \sum_c F_c^2, \quad \mathcal{L}_{\text{AT}} = \|\,\hat{A}(F_s) - \hat{A}(F_t)\,\|_2^2$$

$\hat{A}$: $L_2$ 正規化。チャネル和で次元不一致を解消 → **アダプター不要**。
"どこを見るか（空間位置）"を合わせる — 生値より転移しやすい。


In [6]:
def hinton_kd(s_logits, t_logits, T=4.0):
    return T**2 * F.kl_div(
        F.log_softmax(s_logits / T, -1),
        F.softmax(t_logits / T, -1),
        reduction='batchmean'
    )

def dkd_loss(s_logits, t_logits, T=4.0, alpha=1.0, beta=2.0):
    """Decoupled KD: alpha*TCKD + beta*NCKD."""
    B, C = s_logits.shape
    target  = t_logits.argmax(1)
    nt_mask = torch.ones(B, C, dtype=torch.bool)
    nt_mask[range(B), target] = False           # False = ターゲット位置

    # NCKD: 非ターゲットクラス間の KL
    s_non = s_logits.masked_fill(~nt_mask, float('-inf'))
    t_non = t_logits.masked_fill(~nt_mask, float('-inf'))
    nckd  = T**2 * F.kl_div(F.log_softmax(s_non / T, -1),
                              F.softmax(t_non  / T, -1), reduction='batchmean')

    # TCKD: 2値 {target logit, logsumexp(non-target)}
    s_lse = s_logits.masked_fill(~nt_mask, float('-inf')).logsumexp(1)
    t_lse = t_logits.masked_fill(~nt_mask, float('-inf')).logsumexp(1)
    s_bin = torch.stack([s_logits[range(B), target], s_lse], 1)
    t_bin = torch.stack([t_logits[range(B), target], t_lse], 1)
    tckd  = T**2 * F.kl_div(F.log_softmax(s_bin / T, -1),
                              F.softmax(t_bin  / T, -1), reduction='batchmean')
    return alpha * tckd + beta * nckd

def attention_map(feat):
    """A(F) = ∑_c F_c², L2 正規化して [B, H*W] を返す。"""
    a = (feat ** 2).sum(dim=1).flatten(1)
    return F.normalize(a, p=2, dim=1)

print("損失関数を定義しました")


損失関数を定義しました


## 6. 汎用訓練ループ

各エポックで：
1. `pool_loader` の全バッチに対して教師のロジット・特徴量を計算 → KD 損失 + 特徴蒸留損失
2. `lab_loader` からランダムバッチを取り CE 損失を加算


In [7]:
def train_distill(kd_fn=None, feat_fn=None, adapter=None,
                  alpha=0.9, feat_w=0.0, ce_w=0.1, epochs=60):
    torch.manual_seed(101)
    s = make_student()
    params = list(s.parameters()) + (list(adapter.parameters()) if adapter else [])
    opt = torch.optim.AdamW(params, lr=2e-3, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)

    hist = []
    lab_it = iter(lab_loader)

    for _ in range(epochs):
        s.train()
        for xb, yb in pool_loader:
            with torch.no_grad():
                t_log, t_feat = teacher(xb)
            s_log, s_feat = s(xb)

            parts = []
            if kd_fn   is not None: parts.append(alpha  * kd_fn(s_log, t_log))
            if feat_fn is not None: parts.append(feat_w * feat_fn(s_feat, t_feat, adapter))

            try:
                xb_l, yb_l = next(lab_it)
            except StopIteration:
                lab_it = iter(lab_loader)
                xb_l, yb_l = next(lab_it)
            s_l, _ = s(xb_l)
            parts.append(ce_w * F.cross_entropy(s_l, yb_l))

            loss = sum(parts)
            opt.zero_grad(); loss.backward(); opt.step()

        sch.step()
        hist.append(evaluate(s))

    return s, hist

print("train_distill を定義しました")


train_distill を定義しました


## 7. Method 1 — Scratch（ベースライン）

150枚のラベルのみで CE 損失。教師の知識を一切使わない。


In [8]:
def train_scratch(epochs=60):
    torch.manual_seed(101)
    s = make_student()
    opt = torch.optim.AdamW(s.parameters(), lr=2e-3, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    hist = []
    for _ in range(epochs):
        s.train()
        for xb, yb in lab_loader:
            opt.zero_grad()
            F.cross_entropy(s(xb)[0], yb).backward()
            opt.step()
        sch.step()
        hist.append(evaluate(s))
    return s, hist

student_scratch, hist_scratch = train_scratch()
acc_scratch = hist_scratch[-1]
print(f"Scratch: {acc_scratch:.4f}")


Scratch: 0.9444


## 8. Method 2 — Logit KD（Hinton 2015）

学習プール全体に対して教師のソフトラベルを適用。
$\alpha=0.9$ で KD を優先、$\gamma=0.1$ で CE（ラベル付きのみ）を加算。


In [9]:
student_kd, hist_kd = train_distill(
    kd_fn=hinton_kd,
    alpha=0.9, ce_w=0.1
)
acc_kd = hist_kd[-1]
print(f"Logit KD: {acc_kd:.4f}  (Scratch からの改善: {acc_kd - acc_scratch:+.4f})")


Logit KD: 0.9911  (Scratch からの改善: +0.0467)


## 9. Method 3 — DKD（Decoupled KD, CVPR 2022）

NCKD の重みを $\beta=2$ に増やして dark knowledge を強化。
ロジットのみ使う改良手法として DKD が KD を上回るかを確認する。


In [10]:
dkd_fn = lambda s, t: dkd_loss(s, t, T=4.0, alpha=1.0, beta=2.0)
student_dkd, hist_dkd = train_distill(
    kd_fn=dkd_fn,
    alpha=0.9, ce_w=0.1
)
acc_dkd = hist_dkd[-1]
print(f"DKD:      {acc_dkd:.4f}  (Scratch からの改善: {acc_dkd - acc_scratch:+.4f})")


DKD:      0.9933  (Scratch からの改善: +0.0489)


## 10. Method 4 — FitNets + KD（特徴 MSE）

アダプター $r: 16\text{ch} \to 64\text{ch}$（1×1 conv）を生徒に追加して
$\mathcal{L} = 0.7 \cdot \text{KD} + 0.5 \cdot \text{MSE}(r(F_s), F_t) + 0.1 \cdot \text{CE}$

アダプターは推論時不要（捨てる）。


In [11]:
def fitnets_fn(s_feat, t_feat, adapter):
    return F.mse_loss(adapter(s_feat), t_feat)

fit_adapter = nn.Conv2d(16, 64, kernel_size=1, bias=True)
nn.init.xavier_normal_(fit_adapter.weight, gain=0.1)
nn.init.zeros_(fit_adapter.bias)

student_fit, hist_fit = train_distill(
    kd_fn=hinton_kd, feat_fn=fitnets_fn, adapter=fit_adapter,
    alpha=0.7, feat_w=0.5, ce_w=0.1
)
acc_fit = hist_fit[-1]
print(f"FitNets+KD: {acc_fit:.4f}  (Scratch からの改善: {acc_fit - acc_scratch:+.4f})")


FitNets+KD: 0.9889  (Scratch からの改善: +0.0444)


## 11. Method 5 — AT + KD（Attention Transfer）

チャネルを $\sum_c F_c^2$ で集約して空間的注意マップを作り、L2 正規化後に MSE。
チャネル数の不一致を自動解消 → **アダプター不要**。

$$\mathcal{L} = 0.7 \cdot \text{KD} + 50 \cdot \text{MSE}(\hat{A}(F_s), \hat{A}(F_t)) + 0.1 \cdot \text{CE}$$

AT 損失は L2 正規化済みベクトル間の MSE なので値が小さく、`feat_w=50` でスケール補正。


In [12]:
def at_fn(s_feat, t_feat, adapter=None):
    return F.mse_loss(attention_map(s_feat), attention_map(t_feat))

student_at, hist_at = train_distill(
    kd_fn=hinton_kd, feat_fn=at_fn,
    alpha=0.7, feat_w=50.0, ce_w=0.1
)
acc_at = hist_at[-1]
print(f"AT+KD:     {acc_at:.4f}  (Scratch からの改善: {acc_at - acc_scratch:+.4f})")


AT+KD:     0.9933  (Scratch からの改善: +0.0489)


## 12. 精度比較


In [13]:
results = {
    'Teacher\n(1347 lbl)' : acc_teacher,
    'Scratch\n(150 lbl)'  : acc_scratch,
    'Logit KD'             : acc_kd,
    'DKD'                  : acc_dkd,
    'FitNets\n+KD'        : acc_fit,
    'AT\n+KD'             : acc_at,
}
colors = ['#2196F3', '#9E9E9E', '#FF9800', '#FF5722', '#4CAF50', '#8BC34A']

fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.bar(list(results.keys()), list(results.values()),
              color=colors, edgecolor='k', linewidth=0.5)
for bar, acc in zip(bars, results.values()):
    ax.text(bar.get_x() + bar.get_width()/2, acc + 0.003,
            f'{acc:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylim(0.70, 1.02)
ax.set_ylabel('テスト精度', fontsize=12)
ax.set_title('各手法のテスト精度（生徒: 150枚ラベルのみ）', fontsize=13)
ax.axhline(acc_teacher, color='#2196F3', ls='--', alpha=0.4, label='Teacher ceiling')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('/tmp/accuracy_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n改善幅 (vs Scratch):")
for name, acc in list(results.items())[2:]:
    print(f"  {name:12s}: {acc - acc_scratch:+.4f}")



改善幅 (vs Scratch):
  Logit KD    : +0.0467
  DKD         : +0.0489
  FitNets
+KD : +0.0444
  AT
+KD      : +0.0489


/tmp/ipykernel_44998/4227364570.py:22: UserWarning: Glyph 12486 (\N{KATAKANA LETTER TE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/4227364570.py:22: UserWarning: Glyph 12473 (\N{KATAKANA LETTER SU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/4227364570.py:22: UserWarning: Glyph 12488 (\N{KATAKANA LETTER TO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/4227364570.py:22: UserWarning: Glyph 31934 (\N{CJK UNIFIED IDEOGRAPH-7CBE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/4227364570.py:22: UserWarning: Glyph 24230 (\N{CJK UNIFIED IDEOGRAPH-5EA6}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/4227364570.py:22: UserWarning: Glyph 21508 (\N{CJK UNIFIED IDEOGRAPH-5404}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/4227364570.py:22: UserWarning: Glyph 25163 (\N{CJK UNIFIED IDEOGRAPH-624B}) missing from fo

## 13. 学習曲線

各手法の精度推移。特徴蒸留手法が序盤から速く収束するか確認する。


In [14]:
fig, ax = plt.subplots(figsize=(9, 4.5))
styles = [
    (hist_scratch, 'Scratch',     '#9E9E9E', '-'),
    (hist_kd,      'Logit KD',    '#FF9800', '-'),
    (hist_dkd,     'DKD',         '#FF5722', '--'),
    (hist_fit,     'FitNets+KD',  '#4CAF50', '-'),
    (hist_at,      'AT+KD',       '#8BC34A', '--'),
]
for hist, label, color, ls in styles:
    ax.plot(hist, label=label, color=color, ls=ls, lw=2)

ax.axhline(acc_teacher, color='#2196F3', ls=':', lw=1.5, alpha=0.7, label='Teacher')
ax.set_xlabel('Epoch'); ax.set_ylabel('テスト精度')
ax.set_title('学習曲線')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/learning_curves.png', dpi=100, bbox_inches='tight')
plt.show()


/tmp/ipykernel_44998/1502875042.py:17: UserWarning: Glyph 12486 (\N{KATAKANA LETTER TE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/1502875042.py:17: UserWarning: Glyph 12473 (\N{KATAKANA LETTER SU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/1502875042.py:17: UserWarning: Glyph 12488 (\N{KATAKANA LETTER TO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/1502875042.py:17: UserWarning: Glyph 31934 (\N{CJK UNIFIED IDEOGRAPH-7CBE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/1502875042.py:17: UserWarning: Glyph 24230 (\N{CJK UNIFIED IDEOGRAPH-5EA6}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/1502875042.py:17: UserWarning: Glyph 23398 (\N{CJK UNIFIED IDEOGRAPH-5B66}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/1502875042.py:17: UserWarning: Glyph 32722 (\N{CJK UNIFIED IDEOGRAPH-7FD2}) missing from fo

## 14. CKA（Centered Kernel Alignment）— 表現の類似度

**CKA** は次元数が異なる2つのモデルの特徴空間の類似度を測る指標（0〜1）。

$$\text{CKA}(K, L) = \frac{\text{HSIC}(K, L)}{\sqrt{\text{HSIC}(K,K)\,\cdot\,\text{HSIC}(L,L)}}$$

ここで $K = XX^\top$, $L = YY^\top$（カーネル行列）、HSIC は二重中心化後の内積。

- CKA = 1 → 同一の表現構造
- CKA = 0 → 全く無関係

**特徴蒸留を使うほど生徒の表現が教師に近づくはず**。


In [15]:
@torch.no_grad()
def linear_cka(model_a, model_b, loader=test_loader):
    """カーネル型 Linear CKA（N×N Gram 行列版、次元数不一致でも OK）"""
    model_a.eval(); model_b.eval()
    fa_list, fb_list = [], []
    for xb, _ in loader:
        _, fa = model_a(xb); _, fb = model_b(xb)
        fa_list.append(fa.flatten(1)); fb_list.append(fb.flatten(1))
    X = torch.cat(fa_list).float()   # [N, dA]
    Y = torch.cat(fb_list).float()   # [N, dB]
    X -= X.mean(0); Y -= Y.mean(0)
    K = X @ X.T; L = Y @ Y.T
    # 二重中心化
    def dc(M):
        return M - M.mean(0, keepdim=True) - M.mean(1, keepdim=True) + M.mean()
    Kc = dc(K); Lc = dc(L)
    num = (Kc * Lc).sum()
    den = torch.sqrt((Kc**2).sum() * (Lc**2).sum())
    return (num / (den + 1e-8)).item()

cka_scores = {
    'Scratch'   : linear_cka(teacher, student_scratch),
    'Logit KD'  : linear_cka(teacher, student_kd),
    'DKD'       : linear_cka(teacher, student_dkd),
    'FitNets+KD': linear_cka(teacher, student_fit),
    'AT+KD'     : linear_cka(teacher, student_at),
}
for name, score in cka_scores.items():
    print(f"  CKA(teacher, {name:12s}) = {score:.4f}")

fig, ax = plt.subplots(figsize=(7, 3.5))
clr = ['#9E9E9E', '#FF9800', '#FF5722', '#4CAF50', '#8BC34A']
bars = ax.bar(list(cka_scores.keys()), list(cka_scores.values()),
              color=clr, edgecolor='k', linewidth=0.5)
for bar, val in zip(bars, cka_scores.values()):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.005,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylim(0, 1.05)
ax.set_ylabel('CKA（教師との特徴類似度）', fontsize=11)
ax.set_title('特徴表現はどれだけ教師に近いか？', fontsize=12)
ax.axhline(1.0, color='k', ls='--', alpha=0.3, label='Perfect alignment')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('/tmp/cka.png', dpi=100, bbox_inches='tight')
plt.show()


  CKA(teacher, Scratch     ) = 0.9583
  CKA(teacher, Logit KD    ) = 0.9772
  CKA(teacher, DKD         ) = 0.9793
  CKA(teacher, FitNets+KD  ) = 0.9795
  CKA(teacher, AT+KD       ) = 0.9819


/tmp/ipykernel_44998/2112969824.py:43: UserWarning: Glyph 65288 (\N{FULLWIDTH LEFT PARENTHESIS}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/2112969824.py:43: UserWarning: Glyph 25945 (\N{CJK UNIFIED IDEOGRAPH-6559}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/2112969824.py:43: UserWarning: Glyph 24107 (\N{CJK UNIFIED IDEOGRAPH-5E2B}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/2112969824.py:43: UserWarning: Glyph 12392 (\N{HIRAGANA LETTER TO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/2112969824.py:43: UserWarning: Glyph 12398 (\N{HIRAGANA LETTER NO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/2112969824.py:43: UserWarning: Glyph 29305 (\N{CJK UNIFIED IDEOGRAPH-7279}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/2112969824.py:43: UserWarning: Glyph 24500 (\N{CJK UNIFIED IDEOGRAPH-5FB4}) missing

/tmp/ipykernel_44998/2112969824.py:43: UserWarning: Glyph 34920 (\N{CJK UNIFIED IDEOGRAPH-8868}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/2112969824.py:43: UserWarning: Glyph 29694 (\N{CJK UNIFIED IDEOGRAPH-73FE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/2112969824.py:43: UserWarning: Glyph 12399 (\N{HIRAGANA LETTER HA}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/2112969824.py:43: UserWarning: Glyph 12393 (\N{HIRAGANA LETTER DO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/2112969824.py:43: UserWarning: Glyph 12428 (\N{HIRAGANA LETTER RE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/2112969824.py:43: UserWarning: Glyph 12384 (\N{HIRAGANA LETTER DA}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/2112969824.py:43: UserWarning: Glyph 12369 (\N{HIRAGANA LETTER KE}) missing from font(s) DejaVu San

## 15. t-SNE — 特徴空間の可視化

テストセット全体の特徴マップを 2D に圧縮して可視化する。
教師に近い色の分離パターンを示すほど"よく転移している"。


In [16]:
@torch.no_grad()
def get_features(model):
    model.eval()
    feats, labels = [], []
    for xb, yb in test_loader:
        _, feat = model(xb)
        feats.append(feat.flatten(1).numpy())
        labels.append(yb.numpy())
    return np.vstack(feats), np.concatenate(labels)

models_to_plot = {
    'Teacher'    : teacher,
    'Scratch'    : student_scratch,
    'Logit KD'   : student_kd,
    'FitNets+KD' : student_fit,
    'AT+KD'      : student_at,
}

print("t-SNE 計算中...")
tsne_results = {}
for name, model in models_to_plot.items():
    feats, labels = get_features(model)
    Z = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=500,
             init='pca').fit_transform(feats)
    tsne_results[name] = (Z, labels)
    print(f"  {name} 完了")

cmap = plt.cm.tab10
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, (name, (Z, lbl)) in zip(axes, tsne_results.items()):
    for c in range(10):
        m = lbl == c
        ax.scatter(Z[m, 0], Z[m, 1], color=cmap(c), s=12, alpha=0.8)
    ax.set_title(name, fontsize=11)
    ax.axis('off')
handles = [mpatches.Patch(color=cmap(c), label=str(c)) for c in range(10)]
axes[0].legend(handles=handles, loc='lower left', ncol=2, fontsize=7,
               framealpha=0.8)
plt.suptitle('t-SNE of stage-2 feature maps (テストセット)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('/tmp/tsne.png', dpi=100, bbox_inches='tight')
plt.show()


t-SNE 計算中...


  Teacher 完了


  Scratch 完了


  Logit KD 完了


  FitNets+KD 完了


  AT+KD 完了


/tmp/ipykernel_44998/3201221490.py:40: UserWarning: Glyph 12486 (\N{KATAKANA LETTER TE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/3201221490.py:40: UserWarning: Glyph 12473 (\N{KATAKANA LETTER SU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/3201221490.py:40: UserWarning: Glyph 12488 (\N{KATAKANA LETTER TO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/3201221490.py:40: UserWarning: Glyph 12475 (\N{KATAKANA LETTER SE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/3201221490.py:40: UserWarning: Glyph 12483 (\N{KATAKANA LETTER SMALL TU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/3201221490.py:41: UserWarning: Glyph 12486 (\N{KATAKANA LETTER TE}) missing from font(s) DejaVu Sans.
  plt.savefig('/tmp/tsne.png', dpi=100, bbox_inches='tight')
/tmp/ipykernel_44998/3201221490.py:41: UserWarning: Glyph 12473 (\N{KATAKANA LETTER SU}) m

/tmp/ipykernel_44998/3201221490.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 16. 注意マップの可視化

AT が転移する信号 $A(F) = \sum_c F_c^2$ を確認する。
生徒（AT+KD）の注意マップが教師と似た空間パターンを持つはず。


In [17]:
N_VIS = 6
xb_vis, yb_vis = next(iter(test_loader))
xb_vis = xb_vis[:N_VIS]; yb_vis = yb_vis[:N_VIS]

attention_models = {
    'Teacher'   : teacher,
    'Scratch'   : student_scratch,
    'Logit KD'  : student_kd,
    'FitNets+KD': student_fit,
    'AT+KD'     : student_at,
}

n_rows = len(attention_models) + 1
fig, axes = plt.subplots(n_rows, N_VIS, figsize=(N_VIS * 1.8, n_rows * 1.8))

for j in range(N_VIS):
    axes[0, j].imshow(xb_vis[j, 0].numpy(), cmap='gray', vmin=0, vmax=1)
    axes[0, j].set_title(f'digit {yb_vis[j].item()}', fontsize=9)
    axes[0, j].axis('off')
axes[0, 0].set_ylabel('Input', rotation=0, labelpad=45, fontsize=9, va='center')

for i, (name, model) in enumerate(attention_models.items()):
    with torch.no_grad():
        _, feat = model(xb_vis)
    att = (feat ** 2).sum(1).numpy()   # [N, 8, 8]
    vmax = att.max()
    for j in range(N_VIS):
        axes[i+1, j].imshow(att[j], cmap='inferno', vmin=0, vmax=vmax,
                             interpolation='bilinear')
        axes[i+1, j].axis('off')
    axes[i+1, 0].set_ylabel(name, rotation=0, labelpad=55, fontsize=9, va='center')

plt.suptitle('注意マップ  A(F) = ∑_c F_c²', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('/tmp/attention_maps.png', dpi=100, bbox_inches='tight')
plt.show()


/tmp/ipykernel_44998/1608889815.py:34: UserWarning: Glyph 27880 (\N{CJK UNIFIED IDEOGRAPH-6CE8}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/1608889815.py:34: UserWarning: Glyph 24847 (\N{CJK UNIFIED IDEOGRAPH-610F}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/1608889815.py:34: UserWarning: Glyph 12510 (\N{KATAKANA LETTER MA}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/1608889815.py:34: UserWarning: Glyph 12483 (\N{KATAKANA LETTER SMALL TU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/1608889815.py:34: UserWarning: Glyph 12503 (\N{KATAKANA LETTER PU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_44998/1608889815.py:35: UserWarning: Glyph 27880 (\N{CJK UNIFIED IDEOGRAPH-6CE8}) missing from font(s) DejaVu Sans.
  plt.savefig('/tmp/attention_maps.png', dpi=100, bbox_inches='tight')
/tmp/ipykernel_44998/1608889815.py:35: UserWarning: Glyp

/tmp/ipykernel_44998/1608889815.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 17. まとめ

### なぜ中間特徴量がロジットより有効か

| 観点 | ロジット蒸留 | 特徴量蒸留 |
|---|---|---|
| **情報量** | 10次元（クラス数のみ） | 4,096次元（空間・チャネル・強度） |
| **勾配パス** | head → 全下流層（ボトルネック経由） | 特定 Conv 層に直接 |
| **転移する知識** | クラス間の類似性（dark knowledge） | どこを見るか・何に反応するか |
| **アダプター** | 不要 | FitNets: 1×1 conv が必要、AT: 不要 |
| **困難タスクへの有効性** | 検出・分割ではほぼ効かない | FGD・CWD 等で大きな改善 |

### CKA からわかること

- **FitNets / AT** を使った生徒は Scratch・Logit KD より CKA が高い → 教師に近い表現を学習
- 精度だけでなく**内部表現の質**が向上している

### 設計チートシート

| 状況 | 推奨 |
|---|---|
| ロジットのみ手元にある | **DKD**（NCKD 重みを増やす） |
| 特徴マップにアクセス可能 | **AT**（アダプター不要、安定） |
| dim 差が大きい | **AT** or **CRD**（関係量は次元不一致を無視） |
| 検出・分割タスク | **FGD**（前景集中）/ **CWD**（チャネルワイズ） |
| BERT系 Transformer | **TinyBERT**（Attention 行列 + 隠れ状態） |
| `capacity gap` が大きい | **TAKD**（中間サイズの Teacher Assistant を挟む） |

### 共通の注意事項

1. CE 損失は常に残す（`L = CE + α·KD + β·feat`）
2. 特徴ロスのスケールを CE/KD と合わせる（AT は `feat_w=50` 程度が必要）
3. 教師は必ず `eval()` + `no_grad()` で固定する
4. アダプター（FitNets）は推論時に削除する
5. **capacity gap が大きすぎる場合** は中間サイズの TA を挟む（TAKD）
